# 2. Model Runs
This notebook runs the EuroSAT RemoteCLIP + GDA model step by step. Every run writes a timestamped result folder under `outputs/eurosat/<run-name>/`

If you prefer full run,

```bash
python scripts/run_all.py --config configs/eurosat_benchmark_all.yaml --skip_existing
```

Preset configs:

- `configs/eurosat_zeroshot_clip.yaml`
- `configs/eurosat_clip_gda_tta_supportviews.yaml`
- `configs/eurosat_clip_tip_adapter.yaml`
- `configs/eurosat_remoteclip_gda_tta_supportviews.yaml`
- `configs/eurosat_convnext_tiny.yaml`


## 1. Choose Config


In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "eurosat_remoteclip_gda_tta_supportviews.yaml"
SHOTS = 16
SEED = 42
SAVE_RUN = True
CONFIG_PATH


WindowsPath('c:/Users/user/Documents/projects/oxfordflowers/FewShotEuroSAT/configs/eurosat_remoteclip_gda_tta_supportviews.yaml')

In [6]:
import pandas as pd

from src.utils import get_dataset_name, load_yaml, resolve_device, save_yaml, set_seed, update_config_from_cli

config = update_config_from_cli(load_yaml(CONFIG_PATH), shots=SHOTS, seed=SEED)

# Resolve repo-relative paths so the notebook works from VS Code, JupyterLab,
# or a terminal launched inside either the repository root or notebooks/.
config["experiment"]["data_root"] = str(PROJECT_ROOT / config["experiment"].get("data_root", "data"))
config["experiment"]["output_root"] = str(PROJECT_ROOT / config["experiment"].get("output_root", "outputs"))
config["dataset"]["root"] = str(PROJECT_ROOT / config["dataset"]["root"])
if config.get("clip", {}).get("checkpoint_cache_dir"):
    config["clip"]["checkpoint_cache_dir"] = str(PROJECT_ROOT / config["clip"]["checkpoint_cache_dir"])

exp = config["experiment"]
clip_cfg = config["clip"]
gda_cfg = config["gda"]
dataset_name = get_dataset_name(config)
set_seed(int(exp["seed"]))
device = resolve_device(str(exp.get("device", "auto")))

pd.DataFrame([
    {"setting": "dataset", "value": dataset_name},
    {"setting": "method", "value": exp["method"]},
    {"setting": "shots per class", "value": exp["shots"]},
    {"setting": "seed", "value": exp["seed"]},
    {"setting": "backbone", "value": f"{clip_cfg['pretrained']} / {clip_cfg['model_name']}"},
    {"setting": "device", "value": str(device)},
])


,setting,value
0,dataset,eurosat
1,method,clip_gda
2,shots per class,16
3,seed,42
4,backbone,remoteclip / ViT-B-32
5,device,cuda


## 2. Check RemoteCLIP Checkpoint Readiness

RemoteCLIP is downloaded from Hugging Face on first use. This check runs before creating an output folder, so a missing checkpoint or low disk space does not leave an incomplete run behind.


In [7]:
import shutil

checkpoint_cache_dir = Path(clip_cfg.get("checkpoint_cache_dir", PROJECT_ROOT / "checkpoints" / "remoteclip"))
checkpoint_ready = checkpoint_cache_dir.exists() and any(checkpoint_cache_dir.rglob("*.pt"))
free_bytes = shutil.disk_usage(PROJECT_ROOT).free
required_bytes = 700 * 1024**2

if not checkpoint_ready and free_bytes < required_bytes:
    raise RuntimeError(
        "RemoteCLIP is not cached and there is not enough free disk space to download it. "
        f"Free space: {free_bytes / 1024**2:.1f} MB. Required: about {required_bytes / 1024**2:.0f} MB. "
        "Free disk space or place RemoteCLIP-ViT-B-32.pt under checkpoints/remoteclip, then rerun this notebook."
    )

pd.DataFrame([{
    "checkpoint_cache_dir": str(checkpoint_cache_dir),
    "checkpoint_ready": checkpoint_ready,
    "free_space_mb": round(free_bytes / 1024**2, 1),
}])


,checkpoint_cache_dir,checkpoint_ready,free_space_mb
0,c:\Users\user\Documents\projects\oxfordflowers...,True,527.1


## 3. Load Dataset and Frozen RemoteCLIP

The dataset uses the prepared ImageFolder split. RemoteCLIP is loaded frozen; no backbone weights are updated.


In [8]:
from src.clip_models import load_clip
from src.data import build_datasets
from src.utils import make_run_dir, setup_logger

model, preprocess, tokenizer = load_clip(
    clip_cfg["model_name"],
    clip_cfg.get("pretrained"),
    device,
    checkpoint_path=clip_cfg.get("checkpoint_path"),
    hf_repo_id=clip_cfg.get("hf_repo_id"),
    hf_filename=clip_cfg.get("hf_filename"),
    checkpoint_cache_dir=clip_cfg.get("checkpoint_cache_dir"),
)
train_data, val_data, test_data, class_names = build_datasets(config, transform=preprocess)

run_dir = make_run_dir(exp["output_root"], exp["method"], int(exp["shots"]), int(exp["seed"]), dataset_name)
logger = setup_logger(run_dir)
save_yaml(config, run_dir / "config_used.yaml")

pd.DataFrame([
    {"split": "train", "images": len(train_data)},
    {"split": "val", "images": len(val_data)},
    {"split": "test", "images": len(test_data)},
])


c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\src\clip_models.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, m

,split,images
0,train,18900
1,val,5400
2,test,2700


## 4. Build the Few-Shot Support Set

The support pool is `train + val`. At 16-shot and 10 classes, GDA receives 160 labeled support examples before support view expansion.


In [9]:
from src.fewshot import sample_fewshot_indices_from_targets, save_support_metadata, support_metadata

pool_targets = train_data.targets + val_data.targets
support_by_class = sample_fewshot_indices_from_targets(pool_targets, int(exp["shots"]), int(exp["seed"]))
support_meta = support_metadata(support_by_class, class_names, int(exp["shots"]), int(exp["seed"]), split="train+val")
support_meta["dataset"] = dataset_name
save_support_metadata(run_dir / "support_indices.json", support_meta)

pd.DataFrame([
    {"class_id": cls, "class_name": class_names[cls], "support_count": len(indices)}
    for cls, indices in sorted(support_by_class.items())
])


,class_id,class_name,support_count
0,0,annual crop land,16
1,1,forest,16
2,2,herbaceous vegetation,16
3,3,highway or road,16
4,4,industrial area,16
5,5,pasture,16
6,6,permanent crop land,16
7,7,residential area,16
8,8,river,16
9,9,sea or lake,16


## 5. Extract or Load Frozen Features

RemoteCLIP maps each image into a normalized feature vector. The code uses cached feature files when available, otherwise it extracts features and writes the cache.


In [10]:
import numpy as np

from scripts.run_experiment import cached_clip_features, cached_zeroshot_weights, validation_keep_mask
from src.clip_models import clip_logits, save_logits

batch_size = int(clip_cfg.get("batch_size", 64))
num_workers = int(clip_cfg.get("num_workers", 0))
feature_augmentations = list(clip_cfg.get("feature_augmentations", ["identity"]))

zeroshot_weights = cached_zeroshot_weights(model, tokenizer, class_names, device, exp["data_root"], dataset_name, clip_cfg, logger)
train_pack = cached_clip_features(model, train_data, device, batch_size, num_workers, feature_augmentations, exp["data_root"], dataset_name, clip_cfg, "train", logger)
val_pack = cached_clip_features(model, val_data, device, batch_size, num_workers, feature_augmentations, exp["data_root"], dataset_name, clip_cfg, "val", logger)
test_pack = cached_clip_features(model, test_data, device, batch_size, num_workers, feature_augmentations, exp["data_root"], dataset_name, clip_cfg, "test", logger)

pd.DataFrame([
    {"name": "zero-shot text weights", "shape": tuple(zeroshot_weights.shape)},
    {"name": "train features", "shape": train_pack["features"].shape},
    {"name": "val features", "shape": val_pack["features"].shape},
    {"name": "test features", "shape": test_pack["features"].shape},
])


2026-04-18 22:16:59,336 | INFO | Loading cached CLIP zero-shot text weights: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_zeroshot_promptensemble1_d77ef039.npy


INFO:flowers_gda:Loading cached CLIP zero-shot text weights: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_zeroshot_promptensemble1_d77ef039.npy


2026-04-18 22:16:59,348 | INFO | Loading cached CLIP train features: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_train_aug_41fe6794.npz


INFO:flowers_gda:Loading cached CLIP train features: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_train_aug_41fe6794.npz


2026-04-18 22:16:59,534 | INFO | Loading cached CLIP val features: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_val_aug_41fe6794.npz


INFO:flowers_gda:Loading cached CLIP val features: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_val_aug_41fe6794.npz


2026-04-18 22:16:59,599 | INFO | Loading cached CLIP test features: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_test_aug_41fe6794.npz


INFO:flowers_gda:Loading cached CLIP test features: c:\Users\user\Documents\projects\oxfordflowers\FewShotEuroSAT\data\feature_cache\eurosat\clip_ViT-B-32_remoteclip_test_aug_41fe6794.npz


,name,shape
0,zero-shot text weights,"(512, 10)"
1,train features,"(18900, 512)"
2,val features,"(5400, 512)"
3,test features,"(2700, 512)"


## 6. Evaluate Zero-Shot RemoteCLIP

This is the prompt only decision rule. It gives a baseline before any few shot statistical adaptation.


In [11]:
from src.metrics import compute_metrics, predictions_frame

test_clip_logits = clip_logits(test_pack["features"], zeroshot_weights)
save_logits(test_clip_logits, run_dir / "clip_zeroshot_logits.npy")
zero_metrics, zero_per_class, zero_confusion = compute_metrics(test_clip_logits, test_pack["targets"], class_names)
pd.DataFrame([{
    "model": "Zero-shot RemoteCLIP",
    "top1_accuracy": zero_metrics["top1_accuracy"],
    "top5_accuracy": zero_metrics["top5_accuracy"],
    "macro_f1": zero_metrics["macro_f1"],
}])


,model,top1_accuracy,top5_accuracy,macro_f1
0,Zero-shot RemoteCLIP,0.375926,0.94037,0.342692


## 7. Prepare GDA Support and Validation Features

GDA estimates a mean vector for each class and one shared covariance matrix. The validation split selects the covariance/ridge and the CLIP-GDA ensemble weight.

$$
z_c(x)=x^T\Sigma_\lambda^{-1}\mu_c-\frac{1}{2}\mu_c^T\Sigma_\lambda^{-1}\mu_c+\log\pi_c
$$


In [12]:
pool_features = np.concatenate([train_pack["features"], val_pack["features"]], axis=0)
pool_targets_np = np.concatenate([train_pack["targets"], val_pack["targets"]], axis=0)
support_rows = np.array([idx for indices in support_by_class.values() for idx in indices], dtype=int)
support_features = pool_features[support_rows]
support_targets = pool_targets_np[support_rows]

val_keep = validation_keep_mask(len(train_pack["targets"]), len(val_pack["targets"]), support_rows)
val_features = val_pack["features"][val_keep]
val_targets = val_pack["targets"][val_keep]
val_clip_logits = clip_logits(val_features, zeroshot_weights)

pd.DataFrame([
    {"name": "support features before view expansion", "shape": support_features.shape},
    {"name": "validation features for hyperparameter search", "shape": val_features.shape},
])


,name,shape
0,support features before view expansion,"(160, 512)"
1,validation features for hyperparameter search,"(5367, 512)"


## 8. Expand Support Views

The selected model uses rotations/flips for support examples, increasing the stability of the estimated class distributions.


In [13]:
from scripts.run_experiment import extract_support_view_features

support_feature_augmentations = gda_cfg.get("support_feature_augmentations")
if support_feature_augmentations:
    support_features, support_targets = extract_support_view_features(
        model,
        train_data,
        val_data,
        support_rows,
        len(train_pack["targets"]),
        device,
        batch_size,
        num_workers,
        [str(aug) for aug in support_feature_augmentations],
        logger,
    )
support_features.shape, support_targets.shape


2026-04-18 22:17:05,891 | INFO | Extracting support-view features for augmentation: identity


INFO:flowers_gda:Extracting support-view features for augmentation: identity
CLIP support val features (identity): 100%|██████████| 1/1 [00:00<00:00,  6.44it/s]

2026-04-18 22:17:07,088 | INFO | Extracting support-view features for augmentation: rot90



INFO:flowers_gda:Extracting support-view features for augmentation: rot90
CLIP support val features (rot90): 100%|██████████| 1/1 [00:00<00:00,  6.79it/s]

2026-04-18 22:17:07,747 | INFO | Extracting support-view features for augmentation: rot180


...

2026-04-18 22:17:09,037 | INFO | Extracting support-view features for augmentation: hflip



INFO:flowers_gda:Extracting support-view features for augmentation: hflip
CLIP support val features (hflip): 100%|██████████| 1/1 [00:00<00:00,  6.98it/s]

2026-04-18 22:17:09,680 | INFO | Extracting support-view features for augmentation: vflip



INFO:flowers_gda:Extracting support-view features for augmentation: vflip
CLIP support val features (vflip): 100%|██████████| 1/1 [00:00<00:00,  7.09it/s]


((960, 512), (960,))

## 9. Fit GDA and Select Hyperparameters

The notebook searches the same ridge/covariance/alpha grid as the config, using validation accuracy.


In [14]:
from src.gda import choose_gda_hyperparams, ensemble_logits

gda, ridge, alpha, covariance, gda_scores = choose_gda_hyperparams(
    support_features,
    support_targets,
    val_features,
    val_targets,
    val_clip_logits,
    len(class_names),
    [float(r) for r in gda_cfg.get("ridge_grid", [gda_cfg.get("ridge", 0.1)])],
    [float(a) for a in gda_cfg["alpha_grid"]],
    [str(c) for c in gda_cfg.get("covariance_options", [gda_cfg.get("covariance", "ridge")])],
)
pd.DataFrame([{"covariance": covariance, "ridge": ridge, "alpha": alpha, "support_rows": len(support_targets)}])


,covariance,ridge,alpha,support_rows
0,ridge,0.3,0.9,960


## 10. Evaluate RemoteCLIP + GDA

The final score ensembles standardized zero-shot logits with standardized GDA logits.


In [15]:
test_gda_logits = gda.logits(test_pack["features"])
final_logits = ensemble_logits(test_clip_logits, test_gda_logits, alpha)

metrics, per_class, confusion = compute_metrics(final_logits, test_pack["targets"], class_names)
metrics.update({
    "adaptation": "training-free_gda",
    "alpha": alpha,
    "ridge": ridge,
    "covariance": covariance,
    "dataset": dataset_name,
})

pd.DataFrame([
    {"model": "Zero-shot RemoteCLIP", "top1_accuracy": zero_metrics["top1_accuracy"], "top5_accuracy": zero_metrics["top5_accuracy"], "macro_f1": zero_metrics["macro_f1"]},
    {"model": "RemoteCLIP + GDA", "top1_accuracy": metrics["top1_accuracy"], "top5_accuracy": metrics["top5_accuracy"], "macro_f1": metrics["macro_f1"]},
])


,model,top1_accuracy,top5_accuracy,macro_f1
0,Zero-shot RemoteCLIP,0.375926,0.940370,0.342692
1,RemoteCLIP + GDA,0.901852,0.996296,0.899195


## 11. Save Run Artifacts

Saved outputs are compatible with `scripts/make_paper_assets.py` and `scripts/run_analytics.py`.


In [16]:
from src.metrics import save_metric_outputs
from src.utils import refresh_latest, save_json
from src.visuals import confusion_matrix_plot, per_class_performance_plot

predictions = predictions_frame(final_logits, test_pack["targets"], test_pack["indices"], test_pack["paths"], class_names)
np.save(run_dir / "gda_logits.npy", test_gda_logits)
save_json({
    "alpha": alpha,
    "ridge": ridge,
    "covariance": covariance,
    "joint_validation_scores": gda_scores,
}, run_dir / "gda_stats.json")
save_metric_outputs(metrics, predictions, per_class, confusion, run_dir)
per_class.to_csv(run_dir / "tables" / "per_class_metrics.csv", index=False)
confusion.to_csv(run_dir / "tables" / "confusion_matrix.csv")
confusion_matrix_plot(run_dir / "confusion_matrix.csv", run_dir / "figures")
per_class_performance_plot(run_dir / "per_class_metrics.csv", run_dir / "figures")
refresh_latest(exp["output_root"], run_dir, dataset_name)
run_dir


WindowsPath('c:/Users/user/Documents/projects/oxfordflowers/FewShotEuroSAT/outputs/eurosat/clip_gda_shot16_seed42_20260418_221654')

## 12. Quick Class-Level Inspection


In [17]:
per_class.sort_values(["accuracy", "f1"])[["class_name", "support_count", "accuracy", "f1"]]


,class_name,support_count,accuracy,f1
6,permanent crop land,250,0.816000,0.824242
3,highway or road,250,0.816000,0.886957
0,annual crop land,300,0.843333,0.897163
8,river,250,0.892000,0.856046
2,herbaceous vegetation,300,0.906667,0.870400
5,pasture,200,0.920000,0.845977
9,sea or lake,300,0.930000,0.957118
1,forest,300,0.943333,0.921824
7,residential area,300,0.953333,0.971138
4,industrial area,250,0.988000,0.961089
